# Vanishing & Exploding Gradients in a Vanilla RNN

This notebook demonstrates the vanishing/exploding gradient problem in a plain
(Elman) recurrent neural network. Everything here is deliberately **unfixed**:

- No gradient clipping
- No gating (no LSTM/GRU)
- No orthogonal/identity recurrent initialization tricks
- No normalization, no skip connections

The point is to *see the problem*, not paper over it. The RNN, benchmark tasks,
and plotting helpers live in `rnn.py`, `tasks.py`, and `visualize.py` in this
same folder.

**Contents**
1. What the network looks like (architecture diagram)
2. Direct measurement: gradient norm vs. time-lag (Pascanu et al., 2013)
3. Benchmark: the Adding Problem (Hochreiter & Schmidhuber, 1997)
4. Benchmark: the Copy Memory Task (Hochreiter & Schmidhuber, 1997; Arjovsky et al., 2016)
5. Watching it happen inside real training
6. Takeaways


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from biornn.models.rnn import VanillaRNNCell, AddingRNN, CopyRNN, set_spectral_radius, total_grad_norm
from biornn.tasks import adding_problem, copy_task
from biornn.device import get_device
from biornn.visualize import plot_rnn_diagram, plot_gradient_norms, plot_training_curve

torch.manual_seed(0)
# np.random.seed() does NOT seed np.random.default_rng() (they are separate
# generators), so we thread one explicit seeded Generator through every task
# call below to keep the whole notebook reproducible end to end.
rng = np.random.default_rng(0)

# Training loops run on DEVICE. `BIORNN_DEVICE=cpu` overrides it; see biornn/device.py
# for why the CPU can be faster for these small models.
DEVICE = get_device()
print('training on', DEVICE)

## 1. What does the network look like?

A vanilla RNN reuses the **same** three weight matrices ($W_{xh}$, $W_{hh}$, $W_{hy}$)
at every timestep. Unrolled through time, it is a very deep feedforward network
with tied weights — one layer per timestep. That depth is exactly why gradients
can vanish or explode: backpropagating through $T$ timesteps multiplies together
$T$ copies of (roughly) the same Jacobian.


In [ ]:
fig = plot_rnn_diagram(unroll_steps=5)
plt.show()


## 2. Direct measurement: gradient norm vs. time-lag

*(Pascanu, Mikolov & Bengio, 2013, "On the difficulty of training recurrent
neural networks")*

Before training anything, we can directly see the problem. Take a freshly
initialized RNN cell, run it forward for $T$ steps, and measure
$\left\lVert \partial h_T / \partial h_t \right\rVert$ for every $t$ — i.e. how much a
perturbation at time $t$ affects the final hidden state. We sweep the
**spectral radius** of $W_{hh}$ (its largest eigenvalue magnitude), which controls
whether the network sits in the vanishing ($\rho < 1$) or exploding ($\rho > 1$) regime.


In [ ]:
T = 30
hidden_size = 20
curves = {}

for radius in [0.5, 1.0, 1.5]:
    cell = VanillaRNNCell(1, hidden_size)
    set_spectral_radius(cell, radius)

    h = torch.zeros(1, hidden_size, requires_grad=True)
    hidden_states = [h]
    x = torch.randn(T, 1, 1)
    for t in range(T):
        h = cell(x[t], hidden_states[-1])
        h.retain_grad()
        hidden_states.append(h)

    # backprop from the final hidden state
    hidden_states[-1].pow(2).sum().backward()
    norms = [hs.grad.norm().item() for hs in hidden_states]
    curves[f'spectral radius = {radius}'] = norms[::-1]  # index by lag T-t

fig = plot_gradient_norms(curves)
plt.show()


With $\rho = 0.5$ the gradient norm decays roughly geometrically with lag — the
classic **vanishing** gradient. With $\rho = 1.5$ it stays large or grows over much
of the range — the **exploding** gradient (the curve is noisier because `tanh`
saturation keeps fighting the growth, which is itself part of the story: a vanilla
RNN's nonlinearity caps how explosive things can get *within one step*, but says
nothing about accumulation over many steps). $\rho = 1.0$ sits on the knife's edge.


## 3. Benchmark: the Adding Problem

*(Hochreiter & Schmidhuber, 1997; also used by Pascanu et al., 2013)*

Each input is a length-`seq_len` sequence of `(value, marker)` pairs. Exactly two
timesteps are marked; the target is the sum of the two marked values. Solving this
requires carrying information across the *whole* gap between the two markers — a
direct test of long-range credit assignment through BPTT.

A model that cannot use long-range gradient signal can do no better than always
predicting the unconditional mean of the target ($\approx 1.0$), giving an MSE
around $\mathrm{Var}(U_1+U_2) \approx 0.167$ — that's our baseline to beat.


In [ ]:
seq_lens = [10, 50, 150]
adding_loss_curves = {}
adding_models = {}

for seq_len in seq_lens:
    model = AddingRNN(hidden_size=32).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-2)
    losses = []
    for epoch in range(300):
        x, y = adding_problem(128, seq_len, rng=rng, device=DEVICE)
        opt.zero_grad()
        pred = model(x)
        loss = nn.functional.mse_loss(pred, y)
        loss.backward()          # <-- no clipping here
        opt.step()
        losses.append(loss.item())
    adding_loss_curves[f'seq_len={seq_len}'] = losses
    adding_models[seq_len] = model
    print(f'seq_len={seq_len:>4}  final MSE={losses[-1]:.4f}  (baseline ~0.167)')

fig = plot_training_curve(adding_loss_curves, title='Adding Problem: training loss')
plt.axhline(0.167, color='gray', linestyle='--', linewidth=1)
plt.show()


The short sequence (`seq_len=10`) learns the task easily and drops well below the
baseline. As the sequence lengthens, the gap the gradient has to cross grows, and
training increasingly fails to beat the "just predict the mean" baseline — the
signature of the vanishing gradient problem.

### Now the other failure mode: exploding gradients

If we instead initialize $W_{hh}$ with a large spectral radius and train with
plain SGD (no clipping), a single early step can produce a gradient norm many
orders of magnitude larger than normal, blowing up the parameters in one update.


In [ ]:
torch.manual_seed(1)
exploding_model = AddingRNN(hidden_size=32).to(DEVICE)
set_spectral_radius(exploding_model.cell, 3.0)
opt = torch.optim.SGD(exploding_model.parameters(), lr=1e-2)

grad_norms, losses = [], []
for epoch in range(15):
    x, y = adding_problem(64, 100, rng=rng, device=DEVICE)
    opt.zero_grad()
    pred = exploding_model(x)
    loss = nn.functional.mse_loss(pred, y)
    loss.backward()             # <-- again, no clipping
    grad_norms.append(total_grad_norm(exploding_model))
    opt.step()                  # a NaN/huge gradient corrupts the weights immediately
    losses.append(loss.item())
    print(f'epoch {epoch:>2}  loss={loss.item():.4f}  grad_norm={grad_norms[-1]:.3e}')

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(grad_norms, marker='o')
ax.set_yscale('log')
ax.set_xlabel('epoch')
ax.set_ylabel('total gradient norm (log scale)')
ax.set_title('Exploding gradient: total grad norm per epoch (no clipping)')
plt.show()


Notice the very first gradient norm is many orders of magnitude larger than every
later one. With no clipping, that single update jolts the weights and destabilizes
the loss for several epochs afterward. This is exactly the failure mode gradient
clipping exists to prevent — which we are deliberately not doing here.


## 4. Benchmark: the Copy Memory Task

*(Hochreiter & Schmidhuber, 1997; Arjovsky, Shah & Bengio, 2016)*

A single symbol is shown at $t=0$. After `delay` blank steps and a delimiter, the
network must reproduce that symbol. Unlike the Adding Problem, the input carries
**no new information** during the delay — the only way to succeed is to hold the
symbol in the hidden state across the whole gap, undamaged, which is precisely
what a vanishing gradient prevents the network from learning to do.


In [ ]:
def recall_accuracy(logits, y, pattern_len):
    pred = logits[:, -pattern_len:, :].argmax(dim=-1)
    target = y[:, -pattern_len:]
    return (pred == target).float().mean().item()

pattern_len = 1
n_categories = 8
delays = [0, 20, 60]
copy_loss_curves = {}

for delay in delays:
    model = CopyRNN(hidden_size=32, n_categories=n_categories).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=5e-3)
    losses = []
    for epoch in range(400):
        x, y = copy_task(64, pattern_len, delay, n_categories, rng=rng, device=DEVICE)
        opt.zero_grad()
        logits = model(x)
        loss = nn.functional.cross_entropy(
            logits.reshape(-1, logits.shape[-1]), y.reshape(-1))
        loss.backward()        # <-- no clipping
        opt.step()
        losses.append(loss.item())
    copy_loss_curves[f'delay={delay}'] = losses

    x, y = copy_task(512, pattern_len, delay, n_categories, rng=rng, device=DEVICE)
    with torch.no_grad():
        logits = model(x)
    acc = recall_accuracy(logits, y, pattern_len)
    chance = 1.0 / n_categories
    print(f'delay={delay:>3}  recall accuracy={acc:.3f}  (chance level={chance:.3f})')

fig = plot_training_curve(copy_loss_curves, title='Copy Task: training loss')
plt.show()


With zero delay the RNN learns the task perfectly. As the delay grows, recall
accuracy collapses toward chance level ($1/8 = 0.125$) — the network simply cannot
carry the symbol across a long gap once the gradient signal needed to learn how
to do so has vanished.


## 5. Watching it happen inside real training

Finally, let's connect part 2's abstract gradient-norm-vs-lag curve to the actual
models we just trained on the Adding Problem. For the `seq_len=10` model (which
learned) and the `seq_len=150` model (which didn't), we backprop one real training
batch and look at $\left\lVert \partial \mathcal{L} / \partial h_t \right\rVert$ at every timestep.


In [ ]:
real_curves = {}
for seq_len in [10, 150]:
    model = adding_models[seq_len]
    x, y = adding_problem(32, seq_len, rng=rng, device=DEVICE)
    pred, hidden_states = model(x, return_hidden_states=True)
    loss = nn.functional.mse_loss(pred, y)
    loss.backward()
    norms = [h.grad.norm().item() for h in hidden_states]
    real_curves[f'seq_len={seq_len} ({"learned" if seq_len == 10 else "failed"})'] = norms[::-1]

fig = plot_gradient_norms(
    real_curves,
    title='Real training batch: gradient norm vs. time-lag',
    ylabel=r'$\|\partial \mathcal{L} / \partial h_t\|$',
)
plt.show()


For `seq_len=150`, the gradient norm collapses to (numerically) zero well before
reaching the early timesteps — those parts of the sequence receive **no** learning
signal at all, which is exactly why that model failed to beat the baseline in
Section 3. For `seq_len=10` the gradient stays a comparable order of magnitude
across the whole sequence, which is exactly why that model learned successfully.


## 6. Takeaways

- A vanilla RNN unrolled over $T$ steps is, gradient-wise, a $T$-layer network with
 tied weights — BPTT multiplies together $T$ (roughly similar) Jacobians.
- If the recurrent weight matrix's spectral radius is persistently $<1$, those
 products shrink geometrically with lag: **vanishing gradients**. Long-range
 credit assignment becomes numerically zero, and the network can't learn
 dependencies that span more than a few dozen steps.
- If the spectral radius is persistently $>1$, products can grow explosively:
 **exploding gradients**. Without clipping, a single bad batch can produce a
 gradient norm many orders of magnitude too large and destroy the current weights
 in one optimizer step.
- Both failure modes were reproduced here on two standard synthetic benchmarks
 (Adding Problem, Copy Task) using nothing but a plain `tanh` Elman RNN, standard
 initialization, and standard optimizers — no clipping, gating, or fancy init
 was used anywhere in this notebook, by design.
- The standard fixes (gradient clipping, LSTM/GRU gating, orthogonal/identity
 recurrent initialization, careful normalization) all target one or both of
 these mechanisms directly — none of them are used here on purpose.
